In [1]:
# Import Required Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import io
import base64
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Flatten, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc, accuracy_score, precision_score, recall_score, f1_score
import pymongo
import os
from tensorflow.keras.utils import plot_model
import gc
from tqdm import tqdm
import tensorflow as tf
from numba import cuda
import time

In [2]:
def create_new_version_dir(base_path):
    version = 1
    while True:
        version_dir = os.path.join(base_path, f'v{version}')
        if not os.path.exists(version_dir):
            os.makedirs(version_dir)
            print(f"Created new directory: {version_dir}")
            return version_dir
        version += 1

In [3]:
# Funktion zur Überwachung des Speicherverbrauchs
def check_memory_usage(threshold=90):
    memory = psutil.virtual_memory()
    if memory.percent > threshold:
        print(f"Warnung: Speicherverbrauch bei {memory.percent}%. Programm wird gestoppt.")
        raise MemoryError("Speicherverbrauch zu hoch")

In [4]:
# Load and Merge Data (Assuming MongoDB setup is correct)
def load_data():
    mongo_uri = os.getenv('MONGO_URI', 'mongodb://localhost:27017/fingerprintDB')
    client = pymongo.MongoClient(mongo_uri)
    db = client.get_default_database()
    
    canvassamples = list(db['canvassamples'].find())
    fingerprints = list(db['fingerprints'].find())
    
    canvassamples_df = pd.DataFrame(canvassamples)
    fingerprints_df = pd.DataFrame(fingerprints)
    
    merged_df = pd.merge(canvassamples_df, fingerprints_df, 
                         left_on='fingerprintId', 
                         right_on='_id', 
                         suffixes=('_sample', '_fingerprint'))
    
    return merged_df

merged_df = load_data()
print(f"Gesamtdatensatz enthält {len(merged_df)} Einträge.")

In [5]:
# Preprocess Data

# Function to process images in RGB
def process_image_rgb(base64_str, target_size=(224, 224)):
    try:
        image_data = base64.b64decode(base64_str)
        image = Image.open(io.BytesIO(image_data)).convert('RGB')  # Convert image to RGB
        image = image.resize(target_size)
        image_array = np.array(image) / 255.0  # Normalize to values between 0 and 1
        return image_array
    except Exception as e:
        print(f"Fehler bei der Bildverarbeitung: {e}")
        return None

# Function to extract images and labels from DataFrame
def extract_images_from_df(df, example_user_id):
    images = []
    labels = []
    for _, row in tqdm(df.iterrows(), total=df.shape[0], desc="Lade Bilder"):
        image = process_image_rgb(row['sampleData'])
        if image is not None:
            images.append(image)
            labels.append(1 if row['username'] == example_user_id else 0)
    return np.array(images), np.array(labels)

# Example user ID
example_user_id = 'benutzername_1'

# Create DataFrames for each user
user_ids = merged_df['username'].unique()
user_dfs = {user_id: merged_df[merged_df['username'] == user_id] for user_id in user_ids}

# Sample data for the example user
user_df = user_dfs[example_user_id].sample(n=12000, random_state=42)

# Add negative examples and limit
negative_df = pd.concat([user_dfs[user_id] for user_id in user_ids if user_id != example_user_id])
negative_df = negative_df.sample(n=12000, random_state=42)

# Split into Train/Val/Test sets (70% Train, 20% Val, 10% Test)
train_df, test_df = train_test_split(pd.concat([user_df, negative_df]), test_size=0.1, random_state=42)
train_df, val_df = train_test_split(train_df, test_size=0.2222, random_state=42)  # 0.2222 * 0.9 ≈ 0.2

# Process image data
try:
    X_train, y_train = extract_images_from_df(train_df, example_user_id)
    X_val, y_val = extract_images_from_df(val_df, example_user_id)
    X_test, y_test = extract_images_from_df(test_df, example_user_id)
except Exception as e:
    print(f"Fehler bei der Bilddatenverarbeitung: {e}")

# Free up memory
gc.collect()

In [6]:
import os
from tqdm import tqdm
import tensorflow as tf
from tensorflow.keras.models import load_model
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report, precision_score, recall_score, f1_score, roc_curve, auc
import numpy as np
from numba import cuda

# Function to get the latest version directory
def get_latest_version_dir(base_path):
    version = 1
    latest_version_dir = None
    while True:
        version_dir = os.path.join(base_path, f'v{version}')
        if not os.path.exists(version_dir):
            break
        latest_version_dir = version_dir
        version += 1
    return latest_version_dir

In [7]:
# Function to prepare images from the dataframe
def prepare_images(df, num_samples, example_user_id):
    images = []
    labels = []
    
    # Ensure at least 1000 samples from the example user
    example_user_samples = df[df['username'] == example_user_id].sample(n=1000, random_state=42)
    other_samples = df[df['username'] != example_user_id].sample(n=num_samples, random_state=42)
    
    sampled_df = pd.concat([example_user_samples, other_samples])
    
    for _, row in sampled_df.iterrows():
        image = process_image_rgb(row['sampleData'])
        if image is not None:
            images.append(image)
            labels.append(row['username'])
    
    return np.array(images), np.array(labels)

In [8]:
# Test each model with 1000 samples from random users
num_samples = 1000
base_path = '/ssd'  # Ensure this is the correct base path
latest_version_dir = get_latest_version_dir(base_path)
model_names = ['create_model_1', 'create_model_2', 'create_model_3', 'create_model_4', 'create_model_5']
sample_sizes = [100, 200, 500, 1000, 2000, 5000, 10000]
negative_sample_ratios = [0.5, 1, 2]

results = []
example_user_id = 'benutzername_1'  # Beispiel-Benutzer-ID
batch_size = 8  # Reduce batch size to avoid memory issues

In [9]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from tensorflow.keras.models import load_model
from tqdm import tqdm
from PIL import Image
import io
import base64
import logging

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Function to clear GPU memory
def clear_gpu_memory():
    tf.keras.backend.clear_session()
    tf.compat.v1.reset_default_graph()

# Function to prepare images and labels
def prepare_images(df, example_user_id):
    images = []
    labels = []
    for _, row in df.iterrows():
        image = process_image_rgb(row['sampleData'])
        if image is not None:
            images.append(image)
            labels.append(1 if row['username'] == example_user_id else 0)
    return np.array(images), np.array(labels)

# Function to process image from base64 string
def process_image_rgb(base64_str, target_size=(224, 224)):
    try:
        image_data = base64.b64decode(base64_str)
        image = Image.open(io.BytesIO(image_data)).convert('RGB')
        image = image.resize(target_size)
        image_array = np.array(image) / 255.0
        return image_array
    except Exception as e:
        logging.error(f"Error processing image: {e}")
        return None

# Function to evaluate the model
def evaluate_model(y_true, y_pred, y_prob):
    metrics = {
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, zero_division=0),
        'Recall': recall_score(y_true, y_pred, zero_division=0),
        'F1 Score': f1_score(y_true, y_pred, zero_division=0),
        'ROC AUC': roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else None,
    }
    return metrics

# Function to save results to a CSV file
def save_results_to_csv(results, filename="model_evaluation_results.csv", mode='w'):
    results_df = pd.DataFrame(results)
    results_df.to_csv(filename, index=False, mode=mode, header=(mode == 'w'))
    logging.info(f"Results saved to {filename}")

# Main function to evaluate models for a specific architecture
def evaluate_models_for_architecture(architecture, example_user_id='benutzername_1', num_samples=1000, batch_size=32, model_dir='/ssd/v1'):
    # Load data (implement your own data loading function)
    merged_df = load_data()  # This function must be defined
    print("Daten geladen")
    
    results = []
    actual_predicted_results = []
    
    # Filter models for the specified architecture
    model_files = [f for f in os.listdir(model_dir) if architecture in f]
    print(f"Modelle gefiltert: {model_files}")
    
    # Clear existing result files
    if os.path.exists(f"model_evaluation_results_{architecture}_samplesize_{num_samples}.csv"):
        os.remove(f"model_evaluation_results_{architecture}_samplesize_{num_samples}.csv")
    if os.path.exists(f"actual_predicted_results_{architecture}_samplesize_{num_samples}.csv"):
        os.remove(f"actual_predicted_results_{architecture}_samplesize_{num_samples}.csv")
    
    for model_file in tqdm(model_files, desc=f"Evaluating models for {architecture}"):
        if not model_file.endswith('.h5'):
            continue
        model_path = os.path.join(model_dir, model_file)
        
        try:
            model = load_model(model_path)
            logging.info(f"Loaded model {model_file}")
            print(f"Modell {model_file} geladen")
        except Exception as e:
            logging.error(f"Error loading model {model_file}: {e}")
            print(f"Fehler beim Laden des Modells {model_file}: {e}")
            continue
        
        user_ids = merged_df['username'].unique()
        
        # Ensure exactly num_samples samples from example_user_id
        example_user_df = merged_df[merged_df['username'] == example_user_id].sample(n=num_samples, random_state=42)
        X_test_example_user, y_test_example_user = prepare_images(example_user_df, example_user_id)
        
        y_pred_example_user = []
        y_prob_example_user = []
        current_batch_size = batch_size
        
        while True:
            try:
                with tf.device('/GPU:0'):
                    for i in range(0, len(X_test_example_user), current_batch_size):
                        batch = X_test_example_user[i:i + current_batch_size]
                        batch = tf.convert_to_tensor(batch)
                        y_pred_batch = model.predict(batch)
                        y_pred_example_user.extend(y_pred_batch)
                        y_prob_example_user.extend(y_pred_batch)
                break
            except tf.errors.ResourceExhaustedError:
                logging.warning("GPU memory exhausted, switching to CPU")
                clear_gpu_memory()
                current_batch_size = max(1, current_batch_size // 2)
                if current_batch_size == 1:
                    with tf.device('/CPU:0'):
                        for i in range(0, len(X_test_example_user), current_batch_size):
                            batch = X_test_example_user[i:i + current_batch_size]
                            batch = tf.convert_to_tensor(batch)
                            y_pred_batch = model.predict(batch)
                            y_pred_example_user.extend(y_pred_batch)
                            y_prob_example_user.extend(y_pred_batch)
                    break
        
        y_pred_labels_example_user = [1 if pred > 0.5 else 0 for pred in np.array(y_pred_example_user).flatten()]
        
        # Ensure arrays have the same length
        if len(y_test_example_user) != len(y_pred_labels_example_user):
            logging.warning(f"Mismatched lengths for y_test_example_user and y_pred_labels_example_user")
            print(f"Unterschiedliche Längen für y_test_example_user und y_pred_labels_example_user")
            continue
        
        results_df_example_user = pd.DataFrame({
            'Actual_User': y_test_example_user,
            'Predicted_User': y_pred_labels_example_user
        })
        
        metrics_example_user = evaluate_model(results_df_example_user['Actual_User'], results_df_example_user['Predicted_User'], np.array(y_prob_example_user).flatten())
        metrics_example_user['Model'] = model_file
        metrics_example_user['User'] = example_user_id
        results.append(metrics_example_user)
        
        for actual, predicted in zip(y_test_example_user, y_pred_labels_example_user):
            actual_predicted_results.append((model_file, example_user_id, actual, predicted))
        
        # Evaluate other users
        for user_id in tqdm(user_ids, desc=f"Evaluating users for model {model_file}"):
            if user_id == example_user_id:
                continue
            user_df = merged_df[merged_df['username'] == user_id].sample(n=num_samples, random_state=42)
            X_test_users, y_test_users = prepare_images(user_df, example_user_id)
            
            y_pred = []
            y_prob = []
            current_batch_size = batch_size
            
            while True:
                try:
                    with tf.device('/GPU:0'):
                        for i in range(0, len(X_test_users), current_batch_size):
                            batch = X_test_users[i:i + current_batch_size]
                            batch = tf.convert_to_tensor(batch)
                            y_pred_batch = model.predict(batch)
                            y_pred.extend(y_pred_batch)
                            y_prob.extend(y_pred_batch)
                    break
                except tf.errors.ResourceExhaustedError:
                    logging.warning("GPU memory exhausted, switching to CPU")
                    clear_gpu_memory()
                    current_batch_size = max(1, current_batch_size // 2)
                    if current_batch_size == 1:
                        with tf.device('/CPU:0'):
                            for i in range(0, len(X_test_users), current_batch_size):
                                batch = X_test_users[i:i + current_batch_size]
                                batch = tf.convert_to_tensor(batch)
                                y_pred_batch = model.predict(batch)
                                y_pred.extend(y_pred_batch)
                                y_prob.extend(y_pred_batch)
                        break
            
            y_pred_labels = [1 if pred > 0.5 else 0 for pred in np.array(y_pred).flatten()]
            
            # Ensure arrays have the same length
            if len(y_test_users) != len(y_pred_labels):
                logging.warning(f"Mismatched lengths for y_test_users and y_pred_labels for user {user_id}")
                print(f"Unterschiedliche Längen für y_test_users und y_pred_labels für Benutzer {user_id}")
                continue
            
            results_df = pd.DataFrame({
                'Actual_User': y_test_users,
                'Predicted_User': y_pred_labels
            })
            
            metrics = evaluate_model(results_df['Actual_User'], results_df['Predicted_User'], np.array(y_prob).flatten())
            metrics['Model'] = model_file
            metrics['User'] = user_id
            results.append(metrics)
            
            for actual, predicted in zip(y_test_users, y_pred_labels):
                actual_predicted_results.append((model_file, user_id, actual, predicted))
        
        # Save results after evaluating each model
        save_results_to_csv(results, filename=f"model_evaluation_results_{architecture}_samplesize_{num_samples}.csv", mode='a')
        save_results_to_csv(actual_predicted_results, filename=f"actual_predicted_results_{architecture}_samplesize_{num_samples}.csv", mode='a')
        print("Ergebnisse gespeichert")
        
        # Clear GPU memory after each model evaluation
        clear_gpu_memory()
        logging.info(f"Completed evaluation for model {model_file}")
        print(f"Bewertung für Modell {model_file} abgeschlossen")



In [ ]:
if __name__ == "__main__":
    architecture = 'create_model_1'  # Beispielarchitektur
    num_samples = 100  # Anzahl der Samples pro Benutzer
    evaluate_models_for_architecture(architecture, num_samples=num_samples)

In [ ]:
if __name__ == "__main__":
    architecture = 'create_model_2'  # Beispielarchitektur
    num_samples = 100  # Anzahl der Samples pro Benutzer
    evaluate_models_for_architecture(architecture, num_samples=num_samples)

In [ ]:
if __name__ == "__main__":
    architecture = 'create_model_3'  # Beispielarchitektur
    num_samples = 100  # Anzahl der Samples pro Benutzer
    evaluate_models_for_architecture(architecture, num_samples=num_samples)

In [ ]:
if __name__ == "__main__":
    architecture = 'create_model_4'  # Beispielarchitektur
    num_samples = 100  # Anzahl der Samples pro Benutzer
    evaluate_models_for_architecture(architecture, num_samples=num_samples)

In [ ]:
if __name__ == "__main__":
    architecture = 'create_model_5'  # Beispielarchitektur
    num_samples = 100  # Anzahl der Samples pro Benutzer
    evaluate_models_for_architecture(architecture, num_samples=num_samples)

In [10]:
import os
import time
import tensorflow as tf
from tqdm import tqdm

# Unterdrücke TensorFlow-Ausgaben
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
tf.get_logger().setLevel('ERROR')

if __name__ == "__main__":
    architectures = [f'create_model_{i}' for i in range(1, 6)]  # Architekturen von 1 bis 5
    sample_sizes = [1000, 800, 600, 400, 200]  # Verschiedene Sample-Größen

    total_tasks = len(architectures) * len(sample_sizes)
    task_counter = 0

    for architecture in architectures:
        for num_samples in sample_sizes:
            task_counter += 1
            try:
                start_time = time.time()
                print(f"Evaluating models for architecture {architecture} with {num_samples} samples")
                evaluate_models_for_architecture(architecture, num_samples=num_samples)
            except tf.errors.ResourceExhaustedError:a
                print(f"Resource exhausted for {num_samples} samples, clearing GPU memory and retrying with smaller batch size")
                clear_gpu_memory()
                evaluate_models_for_architecture(architecture, num_samples=num_samples)
            except Exception as e:
                print(f"An error occurred: {e}")
                clear_gpu_memory()
            finally:
                elapsed_time = time.time() - start_time
                remaining_tasks = total_tasks - task_counter
                estimated_time_remaining = (elapsed_time / task_counter) * remaining_tasks
                tqdm.write(f"Progress: {task_counter}/{total_tasks} ({(task_counter / total_tasks) * 100:.2f}%) - Estimated time remaining: {estimated_time_remaining:.2f} seconds")
                tqdm.write(f"Task {task_counter} completed in {elapsed_time:.2f} seconds")
                tqdm.write(f"Clearing GPU memory")
                clear_gpu_memory()

    print("All tasks completed.")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import logging
import os

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Function to load and combine results
def load_and_combine_results(sample_size):
    all_results = []
    expected_columns = ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC AUC', 'Model', 'User']
    
    for i in range(1, 6):
        file_path = f"model_evaluation_results_create_model_{i}_samplesize_{sample_size}.csv"
        print(f"Überprüfe Datei: {file_path}")  # Debugging-Information
        if os.path.exists(file_path):
            results_df = pd.read_csv(file_path)
            logging.info(f"Loaded results for architecture {i}: {results_df.shape}")
            logging.debug(f"Results for architecture {i}:\n{results_df.head()}")
            
            # Check and rename columns if necessary
            if results_df.shape[1] == len(expected_columns):
                results_df.columns = expected_columns
            else:
                logging.warning(f"Unexpected columns in file {file_path}: {results_df.columns.tolist()}")
            
            # Add sample size as a new column
            results_df['Sample_Size'] = sample_size
            
            all_results.append((results_df, file_path))
        else:
            logging.warning(f"File {file_path} does not exist and will be skipped.")
    
    if all_results:
        combined_results_df = pd.concat([df for df, _ in all_results], ignore_index=True)
        logging.info(f"Combined results shape: {combined_results_df.shape}")
        logging.debug(f"Combined results columns: {combined_results_df.columns.tolist()}")
        logging.debug(f"Combined results:\n{combined_results_df.head()}")
        return combined_results_df, [file_path for _, file_path in all_results]
    else:
        logging.error("No result files found.")
        print("No result files found.")  # Debugging-Information
        return pd.DataFrame(), []  # Return an empty DataFrame if no files are found

# Function to shorten model names
def shorten_model_names(df):
    df['Model'] = df['Model'].apply(lambda x: '_'.join(x.split('_')[2:5]))
    return df

# Function to sort models
def sort_models(df):
    df['Model_Sort'] = df['Model'].apply(lambda x: (int(x.split('_')[0]), int(x.split('_')[2])))
    df = df.sort_values(by='Model_Sort').drop(columns='Model_Sort')
    return df

# Function to plot metrics
def plot_metrics(combined_results_df, metrics):
    for metric in metrics:
        if metric in combined_results_df.columns:
            plt.figure(figsize=(14, 10))
            sns.lineplot(data=combined_results_df, x='Model', y=metric, marker='o')
            plt.title(f"Vergleich der Modell-Leistungen: {metric}", fontsize=18)
            plt.xlabel("Modelle", fontsize=16)
            plt.ylabel(metric, fontsize=16)
            plt.xticks(rotation=45, fontsize=12)
            plt.yticks(fontsize=12)
            plt.grid(True)
            plt.tight_layout()
            plt.show()
            logging.debug(f"Plotted {metric} for models")
        else:
            logging.warning(f"Metric {metric} does not exist in the combined results")

# Function to plot boxplots
def plot_boxplots(combined_results_df, metrics):
    for metric in metrics:
        if metric in combined_results_df.columns:
            plt.figure(figsize=(12, 8))
            sns.boxplot(x='Model', y=metric, data=combined_results_df)
            plt.title(f"Verteilung von {metric} über verschiedene Modelle", fontsize=18)
            plt.xlabel("Modelle", fontsize=16)
            plt.ylabel(metric, fontsize=16)
            plt.xticks(rotation=45, fontsize=12)
            plt.yticks(fontsize=12)
            plt.tight_layout()
            plt.show()
            logging.debug(f"Boxplot for {metric}")
        else:
            logging.warning(f"Metric {metric} does not exist in the combined results")

# Function to plot metrics by architecture
def plot_metrics_by_architecture(combined_results_df, metrics):
    architectures = combined_results_df['Model'].apply(lambda x: x.split('_')[0]).unique()
    for architecture in architectures:
        arch_df = combined_results_df[combined_results_df['Model'].str.startswith(architecture)]
        for metric in metrics:
            if metric in arch_df.columns:
                plt.figure(figsize=(14, 10))
                sns.lineplot(data=arch_df, x='Model', y=metric, marker='o')
                plt.title(f"Vergleich der Modell-Leistungen für {architecture}: {metric}", fontsize=18)
                plt.xlabel("Modelle", fontsize=16)
                plt.ylabel(metric, fontsize=16)
                plt.xticks(rotation=45, fontsize=12)
                plt.yticks(fontsize=12)
                plt.grid(True)
                plt.tight_layout()
                plt.show()
                logging.debug(f"Plotted {metric} for {architecture} models")
            else:
                logging.warning(f"Metric {metric} does not exist in the combined results for {architecture}")

# Function to summarize best models
def summarize_best_models(combined_results_df):
    best_models = {}
    metrics = ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC AUC']
    for metric in metrics:
        if metric in combined_results_df.columns:
            best_model = combined_results_df.loc[combined_results_df[metric].idxmax(), 'Model']
            best_models[metric] = best_model
            logging.info(f"Model with the best {metric}: {best_model}")
        else:
            logging.warning(f"Metric {metric} does not exist in the combined results")

    # Print summary of best models
    print("\nZusammenfassung der besten Modelle basierend auf verschiedenen Metriken:")
    for metric, model in best_models.items():
        print(f"Das Modell mit dem besten {metric} ist: {model}")

    # Recommendation based on results
    if len(set(best_models.values())) == 1:
        best_model = next(iter(best_models.values()))
        logging.info("Recommendation: The consistently best model is suitable for deployment.")
        print(f"Empfehlung: Das konsistent beste Modell ({best_model}) ist geeignet für den Einsatz.")
    else:
        logging.info("Recommendation: Different models perform better on different metrics. Choose based on specific requirements.")
        print("Empfehlung: Unterschiedliche Modelle performen unterschiedlich gut je nach Metrik. Wähle basierend auf spezifischen Anforderungen.")

# Function to save results to CSV
def save_results_to_csv(df, filename):
    df.to_csv(filename, index=False)
    logging.info(f"Results saved to {filename}")

# Main function to perform analysis
def main(sample_size):
    combined_results_df, file_paths = load_and_combine_results(sample_size)
    
    if combined_results_df.empty:
        logging.error("No data to analyze.")
        print("No data to analyze.")
        return
    
    # Shorten model names
    combined_results_df = shorten_model_names(combined_results_df)
    
    # Sort models
    combined_results_df = sort_models(combined_results_df)
    
    # Check if 'ROC AUC' column exists and fill NaN values with 0
    if 'ROC AUC' in combined_results_df.columns:
        combined_results_df['ROC AUC'].fillna(0, inplace=True)

    # Display results
    logging.info("Summary of model performances:")
    print("Zusammenfassung der Modell-Leistungen:")
    print(combined_results_df.head())  # Display the first few rows of the combined results

    # Identify best models
    summarize_best_models(combined_results_df)

    # Visualize metrics for all models
    metrics = ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC AUC']
    plot_metrics(combined_results_df, metrics)

    # Boxplots for each metric to illustrate distribution
    plot_boxplots(combined_results_df, metrics)

    # Visualize metrics by architecture
    plot_metrics_by_architecture(combined_results_df, metrics)

    # Save results to CSV
    save_results_to_csv(combined_results_df, f"combined_model_evaluation_results_samplesize_{sample_size}.csv")
if __name__ == "__main__":
    sample_size = 1000  # Beispiel-Sample-Größe
    main(sample_size)
